# 반사율과 화면 색은 같은가?

저장값·반사율·화면 표시의 차이 이해

**할 일** · 같은 Landsat 영상의 표시 범위를 바꾸고 영역 평균과 비교

위에서 아래로 실행. 코드를 바꾸기 전 결과를 예상하고, 실행 결과와 비교

## 실행 준비

CPU 런타임 사용. Earth Engine에 등록한 Cloud 프로젝트와 사용 권한 필요. [설정 안내](https://github.com/steelpl/lecture/blob/main/CE776-climate-change-carbon-water-cycle/GEE_SETUP.md) 참고. 데이터 조회에는 본인 계정 인증 필요

In [ ]:
%pip -q install earthengine-api==1.7.52 geemap==0.39.0

In [ ]:
import ee
import geemap

PROJECT_ID = ""  # Earth Engine에 등록한 본인의 Cloud 프로젝트 ID
if not PROJECT_ID.strip():
    raise ValueError("PROJECT_ID에 본인의 등록된 Cloud 프로젝트 ID 입력")
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

In [ ]:
roi = ee.Geometry.Rectangle([126.80, 37.43, 127.15, 37.70], geodesic=False)

In [ ]:
raw_collection = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                  .filterBounds(roi).filterDate('2020-05-01', '2020-10-01'))
def landsat_sr(image):
    clear = image.select('QA_PIXEL').bitwiseAnd(31).eq(0)
    unsaturated = image.select('QA_RADSAT').eq(0)
    return image.select(['SR_B2','SR_B3','SR_B4','SR_B5']).multiply(0.0000275).add(-0.2).updateMask(clear.And(unsaturated))
sr = raw_collection.map(landsat_sr).median().clip(roi)
means = sr.reduceRegion(ee.Reducer.mean(), roi, 30, maxPixels=2_000_000)

Collection 2 저장값에 배율 0.0000275와 오프셋 −0.2 적용. QA_PIXEL의 fill·dilated cloud·cirrus·cloud·shadow 및 포화 픽셀 제외. RGB = B4·B3·B2

In [ ]:
print('images:', raw_collection.size().getInfo())
print('mean surface reflectance:', means.getInfo())

In [ ]:
Map = geemap.Map()
Map.centerObject(roi, 10)
Map.addLayer(sr, {'bands':['SR_B4','SR_B3','SR_B2'], 'min':0, 'max':0.3}, 'RGB max 0.3')
Map.addLayer(sr, {'bands':['SR_B4','SR_B3','SR_B2'], 'min':0, 'max':0.6}, 'RGB max 0.6')
Map

두 레이어를 번갈아 보시오. 화면 밝기가 변해도 계산한 반사율 평균이 달라지지 않는 이유를 설명하시오. 구름을 제거하지 않은 합성과 비교하려면 무엇을 바꾸어야 하는지 설명하시오.

## 자료와 해석 범위

- [Landsat 8 Collection 2 Level 2](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2)

관측 연도는 고정 사례의 기간. 수업 연도와 무관. 코드는 자료 처리 방법을 보여 주며 지도·수치는 인증 후 조회 결과로 판단